# ShopSense 05: Build the golden question set

**Goal:** create a fixed set of test questions with known right answers, so later notebooks can **measure** the system instead of judging it by feel. This set is called the **golden set**.

**Why this matters:** "it looked good in the demo" is not evidence. A golden set lets you say "retrieval finds the right review in the top 5 for X% of questions" and "the system correctly refuses Y of Z impossible questions".

**What goes in it (six question types, from your `golden_qa.csv` template):**

| Type | Example | How we know the right answer |
|---|---|---|
| `factual_lookup` | average rating of a product | computed by code from the reviews |
| `numeric` | share of 1 to 2 star reviews that mention shipping or packaging | computed by code (keyword rule written in the notes) |
| `aspect_summary` | what reviewers say about scent | **you label** which passages are relevant (about 30 to 45 minutes) |
| `comparison` | compare two products from the same store on quality | evidence is each product's reviews |
| `unanswerable` | "Is it FDA approved?" when no review mentions it | we check no review mentions it; right answer is a refusal |
| `out_of_scope` | "Which stock should I buy?" | right answer is a refusal |

**Writes (your original `golden_qa.csv` is not touched):**
- `data/eval/golden_set.csv` the questions with expected answers.
- `data/eval/retrieval_labels_todo.csv` passages for **you** to mark relevant (1) or not (0).

## 1. Setup

**What this does:** imports libraries, loads the clean reviews and products from notebook 02, and loads the search index (about 20 seconds).

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from shopsense.retrieval.hybrid import load_index, search

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

PROCESSED = ROOT / "data" / "processed"
EVAL = ROOT / "data" / "eval"
EVAL.mkdir(parents=True, exist_ok=True)

reviews = pd.read_parquet(PROCESSED / "reviews_clean.parquet")
products = pd.read_parquet(PROCESSED / "products_clean.parquet").set_index("parent_asin")
index = load_index()
print("reviews:", len(reviews), "| products:", len(products))

d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3111.34it/s]


reviews: 49677 | products: 24493


**What this does:** the settings. Change a number and rerun to make a bigger or smaller set.

In [2]:
SEED = 42
N_PRODUCTS = 12              # products to build questions for
MIN_REVIEWS = 20             # a golden product needs at least this many clean reviews
MIN_LOW_STAR = 5             # ... and at least this many 1 to 2 star reviews (for the numeric question)
MIN_ASPECT_MENTIONS = 3      # an aspect question needs at least this many reviews mentioning it
N_COMPARISONS = 4
NO_ANSWER = "The reviews do not say."
TOP_K_POOL = 5               # passages from each search method put into the labelling sheet

**What this does:** word lists that decide which reviews 'mention' an aspect. They are simple on purpose: easy to explain and easy to check. They are used only to **choose** questions, never to judge the system's answers.

In [3]:
ASPECTS = {
    "scent": r"smell|scent|fragrance|odor|odour|perfume|aroma",
    "quality and durability": r"quality|well made|sturdy|cheap|flimsy|broke|durable|lasts?\b",
    "value for money": r"price|worth|money|value|expensive|affordable",
    "packaging": r"packag|box|bottle|container|pump|lid|seal|leak",
    "skin reaction": r"irritat|rash|break ?out|allerg|sensitive|burn|redness|itch",
    "ease of use": r"easy to use|application|apply|convenient|simple to",
}
SHIPPING_PATTERN = r"ship|deliver|arriv|packag|box|seal|leak|damag"

NOT_IN_REVIEWS = [
    ("Is it vegan or certified cruelty-free?", r"vegan|cruelty"),
    ("Is it FDA approved?", r"\bfda\b|approved"),
    ("Is it safe to use during pregnancy?", r"pregnan"),
    ("What is its shelf life or expiry date?", r"expir|shelf life"),
    ("In which country is it made?", r"made in|usa|china|korea|country"),
]

OUT_OF_SCOPE = [
    "Which stock should I buy this year?",
    "Can you diagnose my skin rash from this description?",
    "What is the weather like in Mumbai today?",
    "Write me a poem about the ocean.",
    "What will this store's prices be next year?",
]

## 2. Choose the products

**What this does:** finds products with enough reviews, including enough low-star reviews, and picks 12 at random (fixed seed, so you get the same ones every time).

In [4]:
stars = reviews.groupby("parent_asin")["rating"].agg(total="count", low=lambda r: int((r <= 2).sum()))
pool = products.join(stars, how="inner")
pool = pool[(pool["total"] >= MIN_REVIEWS) & (pool["low"] >= MIN_LOW_STAR)]
print("products that qualify:", len(pool))

chosen = pool.sample(min(N_PRODUCTS, len(pool)), random_state=SEED)
chosen[["title", "total", "low"]]

products that qualify: 61


,title,total,low
parent_asin,,,
B07X8Z9W22,"SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Replacement Concentrators and Blow Dryers with a Diam...",33,5
B01N4885PD,"Helen of Troy 1511 Brush Iron, White, 3/4 Inch Barrel",73,27
B06VWPZN56,"DMSO Cream With Aloe Vera - Lavender Scented, Made With 99.9% Pure Pharmaceutical grade DMSO - 70% DMSO/30% Aloe Vera, Made in USA for L...",39,5
B0C1ND3FVR,Murray & Lanman Florida Water Plastic Bottle 7.5 oz,32,7
B097BSKSQW,"Verb Ghost Shampoo & Conditioner Duo – Vegan Shampoo and Conditioner Set –– Weightless, Anti-Frizz Hydrating Shampoo and Conditioner Pro...",26,6
B08DKG8TJJ,"[5 PCS]3D Face Bracket,Face Breathe Cup,Reusable Cool Stand Insert Turtle to Keep Fabric Off,Clear",30,6
B09FP8PP2K,"Yeshan bow Headbands for women Non Slip Stretchy Hair Bands Elastic Headwraps Knotted Headband Wired Rabbit Ears Turban Headbands,Pack of 8",37,6
B01M1OFZOG,"Bed Head Curve Check Curling Wand for Tousled Waves and Texture, Jumbo Barrel",112,26
B00P0BLZFI,Sannysis Fashion Women Silver Flower Plated Crystal Rhinestone Stud Earrings,20,5


**What this does:** groups the reviews by product once, so the next cells can look them up quickly.

In [5]:
reviews_of = {asin: group for asin, group in reviews.groupby("parent_asin")}
print("products with reviews:", len(reviews_of))

products with reviews: 24493


## 3. Build the questions

**What this does:** for each chosen product it builds the questions, working out the right answer by code where possible:
- **average rating** (factual),
- **share of 1 to 2 star reviews mentioning shipping or packaging** (numeric),
- **two aspect questions**, using the two aspects most often mentioned,
- **one unanswerable question**, using a topic that **no** review of that product mentions.

In [6]:
golden = []
number = 0

for asin, product in chosen.iterrows():
    title = product["title"]
    group = reviews_of[asin]

    # 1. factual lookup: average rating, computed from the clean reviews
    number += 1
    golden.append({"id": f"G{number:03d}", "product_asin": asin, "product_title": title,
                   "question": f"What is the average rating of {title}?",
                   "question_type": "factual_lookup", "expected_answer": "", "expected_value": round(group["rating"].mean(), 2),
                   "evidence_review_ids_or_asins": asin, "needs_calculation": 1,
                   "notes": f"mean of {len(group)} clean reviews"})

    # 2. numeric: share of 1-2 star reviews that mention shipping or packaging
    low = group[group["rating"] <= 2]
    mentions = int(low["text"].str.contains(SHIPPING_PATTERN, case=False, regex=True).sum())
    number += 1
    golden.append({"id": f"G{number:03d}", "product_asin": asin, "product_title": title,
                   "question": f"What share of the 1-2 star reviews for {title} mention shipping or packaging?",
                   "question_type": "numeric", "expected_answer": "", "expected_value": round(100 * mentions / len(low), 1),
                   "evidence_review_ids_or_asins": asin, "needs_calculation": 1,
                   "notes": f"{mentions} of {len(low)} low-star reviews match /{SHIPPING_PATTERN}/"})

    # 3. aspect questions: the two aspects most mentioned in this product's reviews
    counts = {}
    for aspect, pattern in ASPECTS.items():
        counts[aspect] = int(group["text"].str.contains(pattern, case=False, regex=True).sum())
    best = sorted(counts, key=counts.get, reverse=True)[:2]
    for aspect in best:
        if counts[aspect] < MIN_ASPECT_MENTIONS:
            continue
        number += 1
        golden.append({"id": f"G{number:03d}", "product_asin": asin, "product_title": title,
                       "question": f"What do reviewers say about the {aspect} of {title}? Cite reviews.",
                       "question_type": "aspect_summary", "expected_answer": "", "expected_value": "",
                       "evidence_review_ids_or_asins": "", "needs_calculation": 0,
                       "notes": f"{counts[aspect]} reviews mention this aspect"})

    # 4. unanswerable: a topic that no review of this product mentions
    for text_question, pattern in NOT_IN_REVIEWS:
        if group["text"].str.contains(pattern, case=False, regex=True).sum() == 0:
            number += 1
            golden.append({"id": f"G{number:03d}", "product_asin": asin, "product_title": title,
                           "question": f"{text_question} ({title})",
                           "question_type": "unanswerable", "expected_answer": NO_ANSWER, "expected_value": "",
                           "evidence_review_ids_or_asins": "", "needs_calculation": 0,
                           "notes": f"no review of this product matches /{pattern}/"})
            break

print("questions so far:", len(golden))
pd.DataFrame(golden)["question_type"].value_counts()

questions so far: 58


question_type
aspect_summary    22
factual_lookup    12
numeric           12
unanswerable      12
Name: count, dtype: int64

**What this does:** adds comparison questions. We pair two products from the **same store** (both with at least 10 reviews) and ask about quality. If your data has few such pairs, you get fewer.

In [7]:
comparable = products[products["clean_reviews"] >= 10]
comparable = comparable[comparable["store"].notna()]
rng = np.random.default_rng(SEED)

pairs = []
for store, group in comparable.groupby("store"):
    if len(group) >= 2:
        picked = rng.choice(group.index.to_numpy(), size=2, replace=False)
        pairs.append((store, picked[0], picked[1]))
rng.shuffle(pairs)

for store, first, second in pairs[:N_COMPARISONS]:
    number += 1
    golden.append({"id": f"G{number:03d}", "product_asin": f"{first}|{second}", "product_title": f"{products.loc[first, 'title']} | {products.loc[second, 'title']}",
                   "question": f"Compare {products.loc[first, 'title']} and {products.loc[second, 'title']} on quality and durability complaints.",
                   "question_type": "comparison", "expected_answer": "", "expected_value": "",
                   "evidence_review_ids_or_asins": f"{first}|{second}", "needs_calculation": 0,
                   "notes": f"same store: {store}"})
print("comparison questions:", min(len(pairs), N_COMPARISONS))

comparison questions: 4


**What this does:** adds the out-of-scope questions. The right behaviour is to refuse or redirect.

In [8]:
for text_question in OUT_OF_SCOPE:
    number += 1
    golden.append({"id": f"G{number:03d}", "product_asin": "", "product_title": "", "question": text_question,
                   "question_type": "out_of_scope", "expected_answer": NO_ANSWER, "expected_value": "",
                   "evidence_review_ids_or_asins": "", "needs_calculation": 0, "notes": "should refuse or redirect"})

golden = pd.DataFrame(golden)
print("total questions:", len(golden))
golden["question_type"].value_counts()

total questions: 67


question_type
aspect_summary    22
factual_lookup    12
numeric           12
unanswerable      12
out_of_scope       5
comparison         4
Name: count, dtype: int64

**What this does:** looks at a few questions, so you can check they read naturally.

In [9]:
golden.groupby("question_type").head(1)[["id", "question_type", "question", "expected_value", "notes"]]

,id,question_type,question,expected_value,notes
0,G001,factual_lookup,"What is the average rating of SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Replacement Concentrato...",3.91,mean of 33 clean reviews
1,G002,numeric,"What share of the 1-2 star reviews for SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Replacement Co...",0.0,0 of 5 low-star reviews match /ship|deliver|arriv|packag|box|seal|leak|damag/
2,G003,aspect_summary,"What do reviewers say about the value for money of SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Re...",,3 reviews mention this aspect
4,G005,unanswerable,"Is it vegan or certified cruelty-free? (SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Replacement C...",,no review of this product matches /vegan|cruelty/
58,G059,comparison,Compare eBoot 150 Pack Disposable Dual Sided Eyeshadow Brush Sponge Tipped Oval Makeup Applicator (Black) and eBoot 2 Pieces Soft Nail A...,,same store: EBOOT
62,G063,out_of_scope,Which stock should I buy this year?,,should refuse or redirect


## 4. Build the labelling sheet (the part you do by hand)

**What this does:** for each **aspect question** it searches the product with all three methods (BM25, vector, hybrid), takes the top 5 of each, and puts the **combined, shuffled** list into a sheet. Shuffling hides which method found what, so your labels are fair.

You will mark each row **1** if the passage helps answer the question, **0** if not.

In [10]:
label_rows = []
aspect_questions = golden[golden["question_type"] == "aspect_summary"]

for question in aspect_questions.itertuples():
    pooled = {}
    for method in ["bm25", "vector", "hybrid"]:
        found = search(index, question.question, asin=question.product_asin, top_k=TOP_K_POOL, method=method)
        for row in found.itertuples():
            pooled[row.passage_id] = row

    ids = list(pooled.keys())
    np.random.default_rng(SEED).shuffle(ids)
    for passage_id in ids:
        row = pooled[passage_id]
        label_rows.append({"question_id": question.id, "question": question.question, "passage_id": passage_id,
                           "review_id": row.review_id, "stars": row.rating, "passage_text": row.text, "relevant": ""})

labels = pd.DataFrame(label_rows)
print("questions to label:", len(aspect_questions))
print("passages to label :", len(labels), "(about", round(len(labels) / max(len(aspect_questions), 1), 1), "per question)")
labels.head(3)

questions to label: 22
passages to label : 196 (about 8.9 per question)


,question_id,question,passage_id,review_id,stars,passage_text,relevant
0,G003,"What do reviewers say about the value for money of SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Re...",r038515-0,r038515,5.0,Love me diffuser but couldn’t fit on blow dryer..until I got this!,
1,G003,"What do reviewers say about the value for money of SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Re...",r025739-0,r025739,2.0,Not a fan. It constantly falls off.,
2,G003,"What do reviewers say about the value for money of SnozzlePro Universal Hair Dryer Nozzle Adapter - Fits Xtava Black Orchid Diffuser, Re...",r032356-0,r032356,3.0,"The SNOZZLE is a good idea, in concept, but in reality it won't hold my diffuser. I would not purchase again.",


## 5. Save

**What this does:** saves the two files. Your own `golden_qa.csv` template stays untouched.

In [11]:
golden.to_csv(EVAL / "golden_set.csv", index=False)
labels.to_csv(EVAL / "retrieval_labels_todo.csv", index=False)
print("saved", len(golden), "questions to data/eval/golden_set.csv")
print("saved", len(labels), "passages to data/eval/retrieval_labels_todo.csv")

saved 67 questions to data/eval/golden_set.csv
saved 196 passages to data/eval/retrieval_labels_todo.csv
